In [1]:
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import datasets
from torchvision.transforms import ToTensor, transforms
import matplotlib.pyplot as plt
import numpy as np

import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import json

In [39]:
class LeNet5(nn.Module):

    def __init__(self):
        super().__init__()
    
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)

        self.conv2 = nn.Conv2d(6, 16, 5)

        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d) or isinstance(m, nn.Linear):
                nn.init.kaiming_uniform_(m.weight)

    def forward(self, x):

        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))

        x = torch.flatten(x, 1)

        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))

        x = self.fc3(x)

        return x # cross entropy loss already applies log-softmax internally
    
class LeNet5_Filters(nn.Module):

    def __init__(self):
        super().__init__()

        #increase the number of filters in the convolutional layers to 32 and 64 respectively
        self.conv1 = nn.Conv2d(3, 32, 5)
        self.pool = nn.MaxPool2d(2,2)
        
        self.conv2 = nn.Conv2d(32, 64, 5)

        self.fc1 = nn.Linear(64 * 5 * 5, 120)
        self.fc2 = nn.Linear(120,84)
        self.fc3 = nn.Linear(84,10)

        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d) or isinstance(m, nn.Linear):
                nn.init.kaiming_uniform_(m.weight)

    def forward(self,x):

        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))

        x = torch.flatten(x,1)

        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))

        x = self.fc3(x)

        return x # cross entropy loss already applies log-softmax internally
class LeNet5_Dropout(nn.Module):

    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 32, 5)
        self.pool = nn.MaxPool2d(2,2)
        self.conv2 = nn.Conv2d(32, 64, 5)

        self.fc1 = nn.Linear(64*5*5,120)
        self.fc2 = nn.Linear(120,84)
        self.fc3 = nn.Linear(84,10)

        #dropout layer with 0.5 dropout rate
        self.dropout = nn.Dropout(0.5)

        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d) or isinstance(m, nn.Linear):
                nn.init.kaiming_uniform_(m.weight)

    def forward(self,x):

        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))

        x = torch.flatten(x,1)

        #apply dropout after the first two fully connected layers
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.dropout(F.relu(self.fc2(x)))

        x = self.fc3(x)

        return x # cross entropy loss already applies log-softmax internally

In [45]:
cifar10_lenet = LeNet5()
cifar10_lenet_filters = LeNet5_Filters()
cifar10_lenet_dropout = LeNet5_Dropout()

In [47]:
from torchvision import models
from torchinfo import summary

In [52]:
cifar10_lenet.load_state_dict(torch.load("saved_models/cifar10_lenet.pth"))
cifar10_lenet_filters.load_state_dict(torch.load("saved_models/CIFAR10_model1.pth"))
cifar10_lenet_dropout.load_state_dict(torch.load("saved_models/CIFAR10_model2.pth"))

<All keys matched successfully>

In [58]:
summary(cifar10_lenet, input_size=(1, 3, 32, 32))

Layer (type:depth-idx)                   Output Shape              Param #
LeNet5                                   [1, 10]                   --
├─Conv2d: 1-1                            [1, 6, 28, 28]            456
├─MaxPool2d: 1-2                         [1, 6, 14, 14]            --
├─Conv2d: 1-3                            [1, 16, 10, 10]           2,416
├─MaxPool2d: 1-4                         [1, 16, 5, 5]             --
├─Linear: 1-5                            [1, 120]                  48,120
├─Linear: 1-6                            [1, 84]                   10,164
├─Linear: 1-7                            [1, 10]                   850
Total params: 62,006
Trainable params: 62,006
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.66
Input size (MB): 0.01
Forward/backward pass size (MB): 0.05
Params size (MB): 0.25
Estimated Total Size (MB): 0.31

In [59]:
summary(cifar10_lenet_filters, input_size=(1, 3, 32, 32))

Layer (type:depth-idx)                   Output Shape              Param #
LeNet5_Filters                           [1, 10]                   --
├─Conv2d: 1-1                            [1, 32, 28, 28]           2,432
├─MaxPool2d: 1-2                         [1, 32, 14, 14]           --
├─Conv2d: 1-3                            [1, 64, 10, 10]           51,264
├─MaxPool2d: 1-4                         [1, 64, 5, 5]             --
├─Linear: 1-5                            [1, 120]                  192,120
├─Linear: 1-6                            [1, 84]                   10,164
├─Linear: 1-7                            [1, 10]                   850
Total params: 256,830
Trainable params: 256,830
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 7.24
Input size (MB): 0.01
Forward/backward pass size (MB): 0.25
Params size (MB): 1.03
Estimated Total Size (MB): 1.29

In [60]:
summary(cifar10_lenet_dropout, input_size=(1, 3, 32, 32))

Layer (type:depth-idx)                   Output Shape              Param #
LeNet5_Dropout                           [1, 10]                   --
├─Conv2d: 1-1                            [1, 32, 28, 28]           2,432
├─MaxPool2d: 1-2                         [1, 32, 14, 14]           --
├─Conv2d: 1-3                            [1, 64, 10, 10]           51,264
├─MaxPool2d: 1-4                         [1, 64, 5, 5]             --
├─Linear: 1-5                            [1, 120]                  192,120
├─Dropout: 1-6                           [1, 120]                  --
├─Linear: 1-7                            [1, 84]                   10,164
├─Dropout: 1-8                           [1, 84]                   --
├─Linear: 1-9                            [1, 10]                   850
Total params: 256,830
Trainable params: 256,830
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 7.24
Input size (MB): 0.01
Forward/backward pass size (MB): 0.25
Params size (MB): 1.03
Estimated Tot

In [4]:
import os
import json
import math

for file in os.listdir("saved_history"):
    if file.startswith("CIFAR"):
        with open(os.path.join("saved_history", file), 'r') as f:
            data = json.load(f)

            train_acc = data["train_acc"]
            val_acc = data["val_acc"]

            k_train = max(1, math.ceil(len(train_acc) * 0.01))
            k_val = max(1, math.ceil(len(val_acc) * 0.01))

            top_train = sorted(train_acc, reverse=True)[:k_train]
            top_val = sorted(val_acc, reverse=True)[:k_val]

            print(f"{file}")
            print("Top 1% train acc:", top_train)
            print("Top 1% val acc:", top_val)
            print("Best train acc:", max(train_acc))
            print("Best val acc:", max(val_acc))
            print()

CIFAR100_model.json
Top 1% train acc: [0.7858125]
Top 1% val acc: [0.657]
Best train acc: 0.7858125
Best val acc: 0.657

CIFAR10_lenet_history.json
Top 1% train acc: [0.706375]
Top 1% val acc: [0.5923]
Best train acc: 0.706375
Best val acc: 0.5923

CIFAR10_model1_history.json
Top 1% train acc: [0.88115]
Top 1% val acc: [0.6761]
Best train acc: 0.88115
Best val acc: 0.6761

CIFAR10_model2_history.json
Top 1% train acc: [0.749325]
Top 1% val acc: [0.6834]
Best train acc: 0.749325
Best val acc: 0.6834

CIFAR10_pretrained.json
Top 1% train acc: [0.87985]
Top 1% val acc: [0.6725]
Best train acc: 0.87985
Best val acc: 0.6725

